# 07 — Budget-penalty sweep

**Sweep 4.** Fixed canonical instance (n=16, K=4, p=3). Vary the QUBO
budget-penalty coefficient `A ∈ {0.5, 2, 8, 32}`.

Why this exists: in notebook 03 the most-probable bitstring at A=0.5
(the project default) is `1111111111111111` — i.e. the budget penalty
doesn't dominate, so QAOA happily settles on infeasible all-ones or
single-asset solutions. If P(feasible) and P(optimum) jump as A grows,
the failure of QAOA in 03/04/05/06 is a tuning issue, not a method issue.
If they don't, the failure is deeper.

All values share lam=2.0 and K=4 with the rest of the project.
Results cache to `results/penalty_sweep.json`.

> **Runtime:** ~10 min on Colab CPU from scratch (instant on cache hit).

In [ ]:
# === Bootstrap (Colab + local) ===
import os, urllib.request as _u
exec((open('../scripts/bootstrap.py') if os.path.exists('../scripts/bootstrap.py') else _u.urlopen('https://raw.githubusercontent.com/egil10/fys5419/main/project2/code/scripts/bootstrap.py')).read())

# === Project imports ===
import json
import numpy as np
import pandas as pd

from scripts.colab     import out_dir
from scripts.data      import load_universe
from scripts.portfolio import PortfolioProblem, DEFAULTS
from scripts.classical import brute_force
from scripts.qaoa      import solve, make_hamiltonians
from scripts.metrics   import prob_optimal, prob_feasible, scaled_ratio, gap

RESULTS = out_dir('results')
print(f'Results will be saved to: {RESULTS}')

In [ ]:
# === Canonical instance: n=16, K=4, p=3, lam=2.0 ===
K       = DEFAULTS['K_AT_16']
LAM     = DEFAULTS['lam']
P       = 3
N_SEEDS = 10

A_VALUES = [0.5, 2.0, 8.0, 32.0]   # 0.5 is the project default; 32x sweep
print(f'sweeping A in {A_VALUES} at lam={LAM}, K={K}, p={P}')

In [ ]:
r = load_universe()
cache = RESULTS / 'penalty_sweep.json'

if cache.exists():
    rows = json.loads(cache.read_text())
    print(f'loaded {cache.name}')
else:
    rows = []
    for A in A_VALUES:
        pf = PortfolioProblem(r.mu, r.Sigma, lam=LAM, A=float(A), K=K,
                              tickers=r.tickers)
        bf = brute_force(pf)

        # H_C spectrum endpoints in COST units.
        _, _, _, _, _, HC = make_hamiltonians(pf)
        e_opt   = float(HC.min())
        e_worst = float(HC.max())

        qres = solve(pf, p=P, n_restarts=N_SEEDS, seed=42)

        rows.append({
            'A':           float(A),
            'bf_cost':     bf.cost,
            'bf_bitstring': bf.bitstring,
            'qaoa_energy': float(qres['energy']),
            'ratio':       scaled_ratio(qres['energy'], e_opt, e_worst),
            'gap_rel':     gap(qres['energy'], e_opt),
            'p_optimal':   prob_optimal(qres['probs'], bf.x),
            'p_feasible':  prob_feasible(qres['probs'], pf.n, pf.K),
            'e_opt':       e_opt,
            'e_worst':     e_worst,
            'spread':      e_worst - e_opt,
        })
        print(f'  A={A:>6.2f}: bf={bf.cost:+.4f}  qaoa_E={qres["energy"]:+.4f}  '
              f'ratio={rows[-1]["ratio"]:.4f}  '
              f'P(opt)={rows[-1]["p_optimal"]:.4f}  '
              f'P(feas)={rows[-1]["p_feasible"]:.4f}')

    cache.write_text(json.dumps(rows, indent=2))
    print(f'saved -> {cache.name}')

pd.DataFrame(rows)

In [ ]:
import matplotlib.pyplot as plt
from scripts.plotting import apply_style, PALETTE, title, fig_path
apply_style()

df = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].semilogx(df.A, df.ratio, 'o-', color=PALETTE['blue'], lw=2, ms=8, label='scaled ratio')
axes[0].axhline(1.0, color=PALETTE['red'], ls='--', lw=1.2, label='optimum')
axes[0].set_xlabel(r'budget penalty $A$ (log)')
axes[0].set_ylabel(r'scaled ratio $(E_{worst} - E)/(E_{worst} - E_{opt})$')
axes[0].set_ylim(0.0, 1.05)
title(axes[0], 'QAOA approximation ratio vs penalty', 'expect monotone improvement as A grows')
axes[0].legend(); axes[0].grid(False)

axes[1].semilogx(df.A, df.p_optimal,  'o-', color=PALETTE['red'],        lw=2, ms=8, label='P(optimum)')
axes[1].semilogx(df.A, df.p_feasible, 'o-', color=PALETTE['blue_muted'], lw=2, ms=8, label='P(feasible)')
axes[1].axhline(1.0/(1 << 16), color=PALETTE['charcoal'], ls=':', lw=1, label=r'uniform $1/2^{16}$')
axes[1].set_xlabel(r'budget penalty $A$ (log)')
axes[1].set_ylabel('probability')
title(axes[1], 'Measurement concentration', 'higher = penalty is doing its job')
axes[1].legend(); axes[1].grid(False)

plt.tight_layout()
fig.savefig(fig_path('compare', 'penalty_sweep'), bbox_inches='tight')
plt.show()